In [9]:
import json
import requests
from typing import Any, Dict, List, Optional

from openai import OpenAI

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
MODEL = "gpt-4o-mini"  # 원하면 바꿔도 됨

client = OpenAI()

In [10]:
# -----------------------------
# 1) 실제 API 호출 도구 3개
# -----------------------------
def _get_json(url: str, params: Optional[dict] = None) -> Any:
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()

def get_popular_movies() -> Dict[str, Any]:
    """
    GET /movies
    """
    data = _get_json(f"{BASE_URL}/movies")
    # data는 리스트일 가능성이 큼. 모델이 쓰기 좋게 최소 필드만 정리
    movies = []
    for m in data[:20]:
        movies.append({
            "id": m.get("id"),
            "title": m.get("title"),
            "release_date": m.get("release_date"),
            "vote_average": m.get("vote_average"),
            "overview": m.get("overview"),
        })
    return {"movies": movies}

def get_movie_details(id: int) -> Dict[str, Any]:
    """
    GET /movies/:id
    """
    data = _get_json(f"{BASE_URL}/movies/{id}")
    # 상세는 그대로 주되, 중요한 키만 남겨도 됨 (여긴 원본 최대한 유지)
    return data

def get_similar_movies(id: int) -> Dict[str, Any]:
    """
    GET /movies/:id/similar
    """
    data = _get_json(f"{BASE_URL}/movies/{id}/similar")
    movies = []
    for m in data[:20]:
        movies.append({
            "id": m.get("id"),
            "title": m.get("title"),
            "release_date": m.get("release_date"),
            "vote_average": m.get("vote_average"),
            "overview": m.get("overview"),
        })
    return {"movies": movies}

TOOL_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [11]:
# -----------------------------
# 2) OpenAI tools 스키마
# -----------------------------
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetch the current popular movies list.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetch full details for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Movie id"},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Fetch a list of similar movies for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Movie id"},
                },
                "required": ["id"],
            },
        },
    },
]

In [12]:
# -----------------------------
# 3) 멀티턴 메모리 + 에이전트 루프
# -----------------------------
SYSTEM_PROMPT = """\
You are a Movie Agent. You can use tools to call a movie API.
Rules:
- You must use the provided tools (function calling) when data is needed.
- When listing movies, ALWAYS include each movie's ID clearly so the user can refer to it later.
- If the user asks "similar movies" or "details" but no movie ID is known, ask a short clarifying question.
- Keep answers concise and user-friendly in Korean unless the user asks otherwise.
"""

def run_agent_turn(messages: List[Dict[str, Any]], max_tool_loops: int = 8) -> str:
    """
    One user turn may require multiple tool calls. We loop until the model returns
    a final natural-language answer without tool_calls.
    """
    for _ in range(max_tool_loops):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message

        # 1) 모델이 tool을 호출하지 않으면 => 최종 답변
        tool_calls = getattr(msg, "tool_calls", None)
        if not tool_calls:
            final_text = msg.content or ""
            messages.append({"role": "assistant", "content": final_text})
            print(f"AI: {final_text}")
            return final_text

        # 2) 모델이 tool 호출하면 => assistant 메시지(툴콜 포함)를 messages에 저장
        messages.append({
            "role": "assistant",
            "content": msg.content,  # 보통 None
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in tool_calls
            ],
        })

        # 3) tool_calls 각각 실제 실행 후 role="tool"로 결과를 다시 넣음
        for tc in tool_calls:
            name = tc.function.name
            raw_args = tc.function.arguments or "{}"
            try:
                args = json.loads(raw_args)
            except json.JSONDecodeError:
                args = {}

            fn = TOOL_MAP.get(name)
            if fn is None:
                tool_result = {"error": f"Unknown tool: {name}"}
            else:
                try:
                    tool_result = fn(**args) if args else fn()
                except Exception as e:
                    tool_result = {"error": str(e), "tool": name, "args": args}

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_result, ensure_ascii=False),
            })

    # 루프 제한에 걸리면 안전 종료
    fallback = "죄송해요. 도구 호출 루프가 너무 길어져서 중단했어요. 질문을 조금 더 구체적으로 말해줄래요?"
    messages.append({"role": "assistant", "content": fallback})
    print(f"AI: {fallback}")
    return fallback

In [14]:
print("🎬 Movie Agent ready! (type 'q' or 'quit' to exit)")
messages: List[Dict[str, Any]] = [{"role": "system", "content": SYSTEM_PROMPT,}]

while True:
    message = input("Send a message to the LLM...").strip()
    if message.lower() in ("q", "quit", "exit"):
        break

    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    run_agent_turn(messages)

🎬 Movie Agent ready! (type 'q' or 'quit' to exit)
User: 지금 인기 있는 영화 알려줘
AI: 지금 인기 있는 영화 목록입니다:

1. **Mercy**  
   - ID: 1236153  
   - 개봉일: 2026-01-20  
   - 평점: 7.12  
   - 줄거리: 미래의 한 형사가 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 증명하기 위해 첨단 AI 판사와 대결해야 합니다.

2. **Shelter**  
   - ID: 1290821  
   - 개봉일: 2026-01-28  
   - 평점: 7.0  
   - 줄거리: 외딴 섬에서 자발적인 속박 생활을 하는 한 남자가 소녀를 폭풍에서 구한 후, 과거의 적들과 마주하게 되는 사건을 다룹니다.

3. **The Orphans**  
   - ID: 1244975  
   - 개봉일: 2025-08-20  
   - 평점: 6.11  
   - 줄거리: 고아원에서 자란 두 친구가 의문의 사고로 사랑을 잃고, 갱단의 비밀을 파헤치기 위해 팀을 이루게 됩니다.

4. **The Bluff**  
   - ID: 799882  
   - 개봉일: 2026-02-17  
   - 평점: 5.7  
   - 줄거리: 외딴 섬에서 여주인공이 복수심에 불타고 있는 전 선장을 맞이하게 되며 과거의 비밀을 마주하게 됩니다.

5. **A Woman Scorned**  
   - ID: 1301306  
   - 개봉일: 2025-06-09  
   - 평점: 6.45  
   - 줄거리: 가족과의 주말 휴가 중 테러를 당한 여자가 동생의 복수를 위해 범인들을 쫓아가는 이야기를 다룹니다.

더 궁금한 점이 있으면 말씀해 주세요!
User: Mercy 에 대해 더 알려줘
AI: **영화: Mercy**

- **개봉일**: 2026-01-20
- **장르**: 공상과학, 액션, 스릴러
- **러닝타임**: 99분
- **평점**: 7.1 (투표 수: 538)
- 